In [ ]:
!pip install pandas numpy scikit-learn fairlearn pgmpy

In [ ]:
# fair_bbn_system.py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import os

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

def preprocess_adult_for_fair_bbn(csv_path='/kaggle/input/adult-census-income/adult.csv'):
    """Add Adult Income dataset processing to the second code"""
    df = pd.read_csv(csv_path)
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True)
    df['income'] = np.where(df['income']=='>50K', 1, 0)
    
    # Upsample positives
    more_50k = df[df['income']==1]
    dist = more_50k.groupby(['race','sex']).size().reset_index(name='count')
    dist['up_count'] = (dist['count']*22654/dist['count'].sum()).round().astype(int)
    
    ups = []
    for _, row in dist.iterrows():
        group = more_50k[(more_50k['race']==row['race']) & (more_50k['sex']==row['sex'])]
        ups.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
    df = pd.concat([df[df['income']==0]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    # Sensitive labels
    race_map = {"White": 0,"Black": 1,"Asian-Pac-Islander": 2,"Amer-Indian-Eskimo": 3,"Other": 4}
    sex_map = {"Male": 0,"Female": 1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    
    # Columns
    categorical_cols = ['workclass','education','marital.status','occupation','relationship','native.country']
    numeric_cols = ['age','fnlwgt','education.num','capital.gain','capital.loss','hours.per.week']
    
    # Preprocessor for NN
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    # BBN processing
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    # Split train/test
    X = df.drop(columns=['income','race_label','sex_label'])
    y = df['income'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    
    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    # BBN datasets aligned
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }

BANK_DATA_PATH = '/kaggle/input/uci-ml-repository-bank-marketing/bank/bank-full.csv'

def find_bank_dataset():
    """Return the confirmed Bank Marketing dataset path."""
    if not os.path.exists(BANK_DATA_PATH):
        raise FileNotFoundError(f"Bank dataset not found at {BANK_DATA_PATH}")
    print(f"Found bank dataset at: {BANK_DATA_PATH}")
    return BANK_DATA_PATH

def preprocess_bank_for_fair_bbn(csv_path=None):
    if csv_path is None:
        csv_path = find_bank_dataset()
    
    print(f"Loading bank dataset from: {csv_path}")
    
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path, sep=',')
    
    print(f"Dataset shape: {df.shape}")
    
    target_col = 'y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else 'subscribed'
    if target_col not in df.columns:
        target_col = df.columns[-1]
        print(f"Using {target_col} as target column")
    
    df = df[~df.isin(['unknown', 'Unknown', 'UNKNOWN']).any(axis=1)].reset_index(drop=True)
    
    if df[target_col].dtype == 'object':
        df['y'] = np.where(df[target_col].str.lower().isin(['yes', 'y', 'true', '1']), 1, 0)
    else:
        df['y'] = df[target_col]
    
    positives = df[df['y'] == 1]
    negatives = df[df['y'] == 0]
    print(f"Positives: {len(positives)}, Negatives: {len(negatives)}")
    
    if len(positives) == 0:
        print("Warning: No positive samples found!")
        df.loc[df.head(100).index, 'y'] = 1
        positives = df[df['y'] == 1]
        negatives = df[df['y'] == 0]
    
    dist = positives.groupby(['job', 'marital']).size().reset_index(name='count')
    dist['up_count'] = (dist['count'] * len(negatives) / dist['count'].sum()).round().astype(int)
    
    upsamples = []
    for _, row in dist.iterrows():
        group = positives[(positives['job'] == row['job']) & (positives['marital'] == row['marital'])]
        if len(group) > 0:
            upsamples.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
    
    df = pd.concat([negatives] + upsamples).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    marital_map = {"married": 0, "single": 1, "divorced": 2}
    df['marital_label'] = df['marital'].map(marital_map)
    
    job_map = {job: i for i, job in enumerate(df['job'].unique())}
    df['job_label'] = df['job'].map(job_map)
    
    categorical_cols = ['job', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
    categorical_cols = [col for col in categorical_cols if col in df.columns]
    
    numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
    numeric_cols = [col for col in numeric_cols if col in df.columns]
    
    print(f"Categorical features: {categorical_cols}")
    print(f"Numeric features: {numeric_cols}")
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['marital', 'job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['y', 'marital_label', 'job_label'])
    y = df['y'].values
    sens_marital = df['marital_label']
    sens_job = df['job_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_marital_train, sens_marital_test, sens_job_train, sens_job_test = \
        train_test_split(X, y, sens_marital, sens_job, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    print(f"Final dataset - X_train_nn: {X_train_nn.shape}, bbn_train: {bbn_train.shape}")
    
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_marital_train': sens_marital_train.reset_index(drop=True),
        'sens_marital_test': sens_marital_test.reset_index(drop=True),
        'sens_job_train': sens_job_train.reset_index(drop=True),
        'sens_job_test': sens_job_test.reset_index(drop=True)
    }

def create_synthetic_meps_data(n_samples=10000):
    np.random.seed(seed)
    print("Creating synthetic MEPS-like dataset...")
    
    data = {
        'age': np.random.normal(45, 15, n_samples).astype(int),
        'race': np.random.choice(['White', 'Black', 'Hispanic', 'Other'], n_samples, p=[0.6, 0.15, 0.2, 0.05]),
        'sex': np.random.choice(['Male', 'Female'], n_samples, p=[0.48, 0.52]),
        'income': np.random.lognormal(10, 1, n_samples),
        'education': np.random.choice(['LessThanHighSchool', 'HighSchool', 'SomeCollege', 'College'], 
                                    n_samples, p=[0.1, 0.3, 0.4, 0.2]),
        'health_status': np.random.choice(['Excellent', 'VeryGood', 'Good', 'Fair', 'Poor'], 
                                        n_samples, p=[0.2, 0.3, 0.3, 0.15, 0.05]),
    }
    
    df = pd.DataFrame(data)
    
    utilization_prob = (
        0.1 + 
        0.002 * df['age'] +
        -0.0001 * df['income'] +
        (df['health_status'].map({'Excellent': 0, 'VeryGood': 0.1, 'Good': 0.2, 'Fair': 0.4, 'Poor': 0.6})) +
        (df['race'].map({'White': 0, 'Black': 0.1, 'Hispanic': 0.05, 'Other': 0.02}))
    )
    
    df['utilization'] = np.random.binomial(1, np.clip(utilization_prob, 0, 1))
    
    print(f"Synthetic MEPS data created: {df.shape}")
    print(f"Target distribution: {df['utilization'].value_counts()}")
    
    return df

def preprocess_meps_for_fair_bbn():
    df = create_synthetic_meps_data(8000)
    
    pos_df = df[df['utilization']==1]
    neg_count = len(df[df['utilization']==0])
    
    if len(pos_df) > 0:
        dist = pos_df.groupby(['race','sex']).size().reset_index(name='count')
        dist['up_count'] = (dist['count'] * neg_count / dist['count'].sum()).round().astype(int)
        ups = []
        for _, row in dist.iterrows():
            group = pos_df[(pos_df['race']==row['race']) & (pos_df['sex']==row['sex'])]
            if len(group) > 0:
                ups.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
        df = pd.concat([df[df['utilization']==0]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)

    race_map = {cat: i for i, cat in enumerate(df['race'].unique())}
    sex_map = {cat: i for i, cat in enumerate(df['sex'].unique())}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)

    categorical_cols = [c for c in df.columns if c not in ['utilization','race_label','sex_label'] and df[c].dtype == 'object']
    numeric_cols = [c for c in df.columns if c not in ['utilization','race_label','sex_label'] and df[c].dtype in ['int64', 'float64']]

    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['utilization','race_label','sex_label'])
    y = df['utilization'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']

    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)

    print(f"MEPS preprocessing complete - X_train_nn: {X_train_nn.shape}, bbn_train: {bbn_train.shape}")

    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }

def preprocess_lawschool_for_fair_bbn():
    print("==============================================================")
    print("PROCESSING LAW SCHOOL ADMISSIONS + BAR PASSAGE DATASET")
    print("==============================================================")

    law_path = "/kaggle/input/law-school-admissions-bar-passage/bar_pass_prediction.csv"
    law_df = pd.read_csv(law_path)

    print(f"Shape: {law_df.shape}")
    print(f"Columns: {list(law_df.columns)}")

    law_df.columns = [c.strip().lower() for c in law_df.columns]

    target_col = 'pass_bar'
    sens_race = 'race'
    sens_gender = 'sex'

    law_df = law_df.dropna(subset=[target_col, sens_race, sens_gender])

    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = law_df[col].fillna(law_df[col].median())

    law_df['income'] = np.where(law_df['fam_inc'] > law_df['fam_inc'].median(), 1, 0)
    
    high_income = law_df[law_df['income'] == 1]
    dist = high_income.groupby([sens_race, sens_gender]).size().reset_index(name='count')
    target_count = len(law_df[law_df['income'] == 0])
    dist['up_count'] = (dist['count'] * target_count / dist['count'].sum()).round().astype(int)
    
    ups = []
    for _, row in dist.iterrows():
        group = high_income[(high_income[sens_race] == row[sens_race]) & (high_income[sens_gender] == row[sens_gender])]
        if len(group) > 0:
            n_samples = int(row['up_count'])
            ups.append(resample(group, replace=True, n_samples=n_samples, random_state=42))
    
    if ups:
        law_df = pd.concat([law_df[law_df['income'] == 0]] + ups).sample(frac=1, random_state=42).reset_index(drop=True)
    else:
        law_df = law_df.sample(frac=1, random_state=42).reset_index(drop=True)

    law_df['race_label'] = law_df[sens_race].astype('category').cat.codes
    law_df['gender_label'] = law_df[sens_gender].astype('category').cat.codes

    numeric_cols = ['lsat', 'ugpa', 'zfygpa', 'zgpa', 'age', 'gpa', 'fam_inc']
    categorical_cols = ['tier', 'cluster', 'fulltime', 'parttime']

    numeric_cols = [col for col in numeric_cols if col in law_df.columns]
    categorical_cols = [col for col in categorical_cols if col in law_df.columns]

    for col in numeric_cols:
        law_df[col] = law_df[col].fillna(law_df[col].median())

    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

    bbn_df = law_df.copy()
    for col in numeric_cols:
        if len(bbn_df[col].unique()) > 1:
            try:
                bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop')
                bbn_df[col] = bbn_df[col].fillna(0).astype(int)
            except:
                bbn_df[col] = pd.cut(bbn_df[col], 5, labels=False, duplicates='drop')
                bbn_df[col] = bbn_df[col].fillna(0).astype(int)
        else:
            bbn_df[col] = 0
    
    for col in categorical_cols + [sens_race, sens_gender]:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = law_df[numeric_cols + categorical_cols + [sens_race, sens_gender]]
    y = law_df['income'].values
    sens_race_labels = law_df['race_label']
    sens_gender_labels = law_df['gender_label']

    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race_labels, sens_gender_labels, test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)

    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)

    print(f"Final dataset - X_train_nn: {X_train_nn.shape}, bbn_train: {bbn_train.shape}")

    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_gender_train': sens_gender_train.reset_index(drop=True),
        'sens_gender_test': sens_gender_test.reset_index(drop=True)
    }

def preprocess_compas_for_fair_bbn(csv_path='/kaggle/input/compas/compas-scores-two-years.csv', seed=42):
    """Add COMPAS dataset processing from the first code"""
    
    # Load and filter COMPAS data
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex',
         'priors_count','days_b_screening_arrest','decile_score',
         'juv_other_count','juv_misd_count','juv_fel_count',
         'c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    
    # Sensitive label mapping
    race_map = {"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sex_map = {"Male":0,"Female":1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    
    # Jail time calculation
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days
    df['jail_time'] = df['jail_time'].fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    
    # Target variable
    y = df['two_year_recid'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    
    # Columns
    numeric_cols = ['age','priors_count','days_b_screening_arrest','decile_score',
                    'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    
    # Preprocessor for NN (scaled + one-hot)
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    # BBN preprocessing: discretize numerics, encode categoricals
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    # Split train/test
    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label'])
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    
    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    # BBN aligned splits
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }


class SimpleFairBBN:
    def __init__(self, feature_names=None, target_name='y'):
        self.feature_names = feature_names or []
        self.target_name = target_name
        self.model = None
        self.inference = None

    def fit(self, df_discrete, y, include_sensitive=True):
        data = df_discrete.copy().reset_index(drop=True)
        data[self.target_name] = y
        candidate_features = list(df_discrete.columns)
        selected = candidate_features[:6]
        if include_sensitive:
            for sens in ['marital', 'job', 'race', 'sex']:
                if sens in df_discrete.columns and sens not in selected:
                    selected.append(sens)
        edges = [(f, self.target_name) for f in selected]
        self.feature_names = selected
        self.model = DiscreteBayesianNetwork(edges)
        self.model.fit(data, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=5)
        self.inference = VariableElimination(self.model)

    def predict_proba(self, df_discrete):
        Xdf = df_discrete.reset_index(drop=True)
        probs = []
        for _, row in Xdf.iterrows():
            evidence = {}
            used = 0
            for f in self.feature_names:
                if f in row and not pd.isna(row[f]) and used < 3:
                    try:
                        evidence[f] = int(row[f])
                        used += 1
                    except:
                        continue
            try:
                q = self.inference.query(variables=[self.target_name], evidence=evidence) if evidence else self.inference.query(variables=[self.target_name])
                p1 = q.values[1] if len(q.values) == 2 else 0.5
            except:
                p1 = 0.5
            probs.append(p1)
        return np.array(probs)

class AdapterNN(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 1)

    def forward(self, x, return_features=False):
        h = self.act(self.fc1(x))
        h2 = self.act(self.fc2(h))
        logit = self.out(h2)
        return (logit, h2) if return_features else logit

class AdversaryNN(nn.Module):
    def __init__(self, in_dim, marital_classes, job_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.marital_head = nn.Linear(32, marital_classes)
        self.job_head = nn.Linear(32, job_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.marital_head(s), self.job_head(s)

class AdversaryNN_MEPS(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_LawSchool(nn.Module):
    def __init__(self, in_dim, race_classes, gender_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.gender_head = nn.Linear(32, gender_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.gender_head(s)

class AdversaryNN_Adult(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_COMPAS(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

def train_fair_bbn_adapter_compas(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    """Training function for COMPAS dataset"""
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_race_train, sens_race_test = data_dict['sens_race_train'], data_dict['sens_race_test']
    sens_sex_train, sens_sex_test = data_dict['sens_sex_train'], data_dict['sens_sex_test']

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns), target_name='two_year_recid')
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)
    
    # Generate marginals for adapter input
    p_all = bbn.predict_proba(bbn_train_df).reshape(-1,1)
    p_race = bbn.predict_proba(bbn_train_df[['race']]).reshape(-1,1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex']]).reshape(-1,1)
    adapter_in_train = torch.tensor(np.hstack([p_all,p_race,p_sex]), dtype=torch.float32)
    
    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1,1)
    p_race_test = bbn.predict_proba(bbn_test_df[['race']]).reshape(-1,1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex']]).reshape(-1,1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test,p_race_test,p_sex_test]), dtype=torch.float32)

    # Convert labels & sensitive attrs
    y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
    y_test_t  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    sex_train_t  = torch.tensor(sens_sex_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(adapter_in_train, y_train_t, race_train_t, sex_train_t),
        batch_size=batch_size, shuffle=True
    )

    # Initialize adapter + adversary
    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN_COMPAS(in_dim=32,
                            race_classes=len(sens_race_train.unique()),
                            sex_classes=len(sens_sex_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    # Training loop
    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss=0.0; total_adv_loss=0.0
        for batch_in, batch_y, batch_race, batch_sex in train_loader:
            # --- Train adversary ---
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, s_logits = adversary(features)
            adv_loss = (adv_loss_fn(r_logits,batch_race) + adv_loss_fn(s_logits,batch_sex))/2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()
    
            # --- Train adapter ---
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit,batch_y)
            r_logits2, s_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2,batch_race) + adv_loss_fn(s_logits2,batch_sex))/2.0
            # fairness penalty across race groups
            dp_penalty = torch.abs(features[batch_race==0].mean() - features[batch_race==1].mean())
            total_loss = pred_loss - lambda_adv*adv_loss_for_adapter + alpha_bbn*dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()
    
        if epoch % 10==0 or epoch==epochs-1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")
    
    # Evaluation
    adapter.eval()
    with torch.no_grad():
        test_logit,_ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs>0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)

    # Print results
    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Sex  DP: {base_sex_dp:.4f}, Sex  EO: {base_sex_eo:.4f}")
    
    print("\nBBN + Adapter (Adversarial + Fairness) RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Sex  DP: {adv_sex_dp:.4f}, Sex  EO: {adv_sex_eo:.4f}")
    
    return {
        'baseline':{
            'pred':base_pred,'acc':base_acc,
            'race_dp':base_race_dp,'race_eo':base_race_eo,
            'sex_dp':base_sex_dp,'sex_eo':base_sex_eo
        },
        'bbn_adapter':{
            'pred':test_pred,'acc':adv_acc,
            'race_dp':adv_race_dp,'race_eo':adv_race_eo,
            'sex_dp':adv_sex_dp,'sex_eo':adv_sex_eo
        }
    }
    

def train_fair_bbn_adapter_adult(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    """Training function for Adult dataset"""
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_race_train, sens_race_test = data_dict['sens_race_train'], data_dict['sens_race_test']
    sens_sex_train, sens_sex_test = data_dict['sens_sex_train'], data_dict['sens_sex_test']
    
    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns))
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)

    # Generate BBN marginals
    p_all = bbn.predict_proba(bbn_train_df).reshape(-1,1)
    p_race = bbn.predict_proba(bbn_train_df[['race']]).reshape(-1,1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex']]).reshape(-1,1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_race, p_sex]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1,1)
    p_race_test = bbn.predict_proba(bbn_test_df[['race']]).reshape(-1,1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex']]).reshape(-1,1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_race_test, p_sex_test]), dtype=torch.float32)

    # Convert to tensors
    y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
    y_test_t = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    sex_train_t = torch.tensor(sens_sex_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, race_train_t, sex_train_t), 
                              batch_size=batch_size, shuffle=True)

    # Models - USING ADULT-SPECIFIC ADVERSARY
    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN_Adult(in_dim=32, race_classes=len(sens_race_train.unique()), sex_classes=len(sens_sex_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss=0.0; total_adv_loss=0.0
        for batch in train_loader:
            batch_in, batch_y, batch_race, batch_sex = batch

            # Train adversary
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, s_logits = adversary(features)
            adv_loss = (adv_loss_fn(r_logits,batch_race) + adv_loss_fn(s_logits,batch_sex))/2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss+=adv_loss.item()

            # Train adapter
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit,batch_y)
            r_logits2, s_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2,batch_race) + adv_loss_fn(s_logits2,batch_sex))/2.0

            # BBN fairness regularization
            dp_penalty = torch.abs(features[batch_race==0].mean() - features[batch_race==1].mean())
            total_loss = pred_loss - lambda_adv*adv_loss_for_adapter + alpha_bbn*dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10==0 or epoch==epochs-1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")

    # Evaluation
    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit,_ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs>0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Sex  DP: {base_sex_dp:.4f}, Sex  EO: {base_sex_eo:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Sex  DP: {adv_sex_dp:.4f}, Sex  EO: {adv_sex_eo:.4f}")

    return {
        'baseline': {'pred': base_pred, 'acc': base_acc, 'race_dp': base_race_dp, 'race_eo': base_race_eo,
                     'sex_dp': base_sex_dp, 'sex_eo': base_sex_eo},
        'bbn_adapter': {'pred': test_pred, 'acc': adv_acc, 'race_dp': adv_race_dp, 'race_eo': adv_race_eo,
                        'sex_dp': adv_sex_dp, 'sex_eo': adv_sex_eo}
    }


def train_fair_bbn_adapter_bank(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_marital_train, sens_marital_test = data_dict['sens_marital_train'], data_dict['sens_marital_test']
    sens_job_train, sens_job_test = data_dict['sens_job_train'], data_dict['sens_job_test']

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_marital_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_marital_test)
    base_marital_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_marital_test)
    base_job_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_job_test)
    base_job_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_job_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    features_to_use = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'marital', 'job']
    bbn_train_sub = bbn_train_df[features_to_use]
    bbn_test_sub = bbn_test_df[features_to_use]

    bbn = SimpleFairBBN(feature_names=features_to_use, target_name='y')
    bbn.fit(bbn_train_sub, y_train, include_sensitive=True)

    p_all = bbn.predict_proba(bbn_train_sub).reshape(-1, 1)
    p_marital = bbn.predict_proba(bbn_train_sub[['marital']]).reshape(-1, 1)
    p_job = bbn.predict_proba(bbn_train_sub[['job']]).reshape(-1, 1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_marital, p_job]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_sub).reshape(-1, 1)
    p_marital_test = bbn.predict_proba(bbn_test_sub[['marital']]).reshape(-1, 1)
    p_job_test = bbn.predict_proba(bbn_test_sub[['job']]).reshape(-1, 1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_marital_test, p_job_test]), dtype=torch.float32)

    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    y_test_t = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)
    marital_train_t = torch.tensor(sens_marital_train.values.astype(int), dtype=torch.long)
    job_train_t = torch.tensor(sens_job_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, marital_train_t, job_train_t),
                              batch_size=batch_size, shuffle=True)

    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN(in_dim=32, marital_classes=len(sens_marital_train.unique()), job_classes=len(sens_job_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss = 0.0; total_adv_loss = 0.0

        for batch in train_loader:
            batch_in, batch_y, batch_marital, batch_job = batch

            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            m_logits, j_logits = adversary(features)
            adv_loss = (adv_loss_fn(m_logits, batch_marital) + adv_loss_fn(j_logits, batch_job)) / 2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()

            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)
            m_logits2, j_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(m_logits2, batch_marital) + adv_loss_fn(j_logits2, batch_job)) / 2.0

            grp0_mean = features[batch_marital == 0].mean(dim=0)
            grp1_mean = features[batch_marital == 1].mean(dim=0)
            dp_penalty = torch.abs(grp0_mean - grp1_mean).mean()

            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss / len(train_loader):.4f} | Adapter Loss: {total_adapter_loss / len(train_loader):.4f}")

    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit, _ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_marital_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_marital_test)
    adv_marital_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_marital_test)
    adv_job_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_job_test)
    adv_job_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_job_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Marital DP: {base_marital_dp:.4f}, Marital EO: {base_marital_eo:.4f}")
    print(f" Job     DP: {base_job_dp:.4f}, Job     EO: {base_job_eo:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Marital DP: {adv_marital_dp:.4f}, Marital EO: {adv_marital_eo:.4f}")
    print(f" Job     DP: {adv_job_dp:.4f}, Job     EO: {adv_job_eo:.4f}")

    return {
        'baseline': {'pred': base_pred, 'acc': base_acc, 'marital_dp': base_marital_dp, 'marital_eo': base_marital_eo,
                     'job_dp': base_job_dp, 'job_eo': base_job_eo},
        'bbn_adapter': {'pred': test_pred, 'acc': adv_acc, 'marital_dp': adv_marital_dp, 'marital_eo': adv_marital_eo,
                        'job_dp': adv_job_dp, 'job_eo': adv_job_eo}
    }

def train_fair_bbn_adapter_meps(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_race_train, sens_race_test = data_dict['sens_race_train'], data_dict['sens_race_test']
    sens_sex_train, sens_sex_test = data_dict['sens_sex_train'], data_dict['sens_sex_test']

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns), target_name='y')
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)

    p_all = bbn.predict_proba(bbn_train_df).reshape(-1, 1)
    p_race = bbn.predict_proba(bbn_train_df[['race']]).reshape(-1, 1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex']]).reshape(-1, 1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_race, p_sex]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1, 1)
    p_race_test = bbn.predict_proba(bbn_test_df[['race']]).reshape(-1, 1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex']]).reshape(-1, 1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_race_test, p_sex_test]), dtype=torch.float32)

    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    y_test_t = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    sex_train_t = torch.tensor(sens_sex_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, race_train_t, sex_train_t),
                              batch_size=batch_size, shuffle=True)

    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN_MEPS(in_dim=32, race_classes=len(sens_race_train.unique()), sex_classes=len(sens_sex_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss = 0.0; total_adv_loss = 0.0

        for batch in train_loader:
            batch_in, batch_y, batch_race, batch_sex = batch

            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, s_logits = adversary(features)
            adv_loss = (adv_loss_fn(r_logits, batch_race) + adv_loss_fn(s_logits, batch_sex)) / 2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()

            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)
            r_logits2, s_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2, batch_race) + adv_loss_fn(s_logits2, batch_sex)) / 2.0

            grp0_mean = features[batch_race == 0].mean(dim=0)
            grp1_mean = features[batch_race == 1].mean(dim=0)
            dp_penalty = torch.abs(grp0_mean - grp1_mean).mean()

            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss / len(train_loader):.4f} | Adapter Loss: {total_adapter_loss / len(train_loader):.4f}")

    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit, _ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Sex  DP: {base_sex_dp:.4f}, Sex  EO: {base_sex_eo:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Sex  DP: {adv_sex_dp:.4f}, Sex  EO: {adv_sex_eo:.4f}")

    return {
        'baseline': {'pred': base_pred, 'acc': base_acc, 'race_dp': base_race_dp, 'race_eo': base_race_eo,
                     'sex_dp': base_sex_dp, 'sex_eo': base_sex_eo},
        'bbn_adapter': {'pred': test_pred, 'acc': adv_acc, 'race_dp': adv_race_dp, 'race_eo': adv_race_eo,
                        'sex_dp': adv_sex_dp, 'sex_eo': adv_sex_eo}
    }

def train_fair_bbn_adapter_lawschool(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']
    sens_race_train, sens_race_test = data_dict['sens_race_train'], data_dict['sens_race_test']
    sens_gender_train, sens_gender_test = data_dict['sens_gender_train'], data_dict['sens_gender_test']

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_gender_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_gender_test)
    base_gender_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_gender_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    features_to_use = ['lsat', 'ugpa', 'fulltime', 'fam_inc', 'age', 'race', 'gender']
    features_to_use = [f for f in features_to_use if f in bbn_train_df.columns]
    bbn_train_sub = bbn_train_df[features_to_use]
    bbn_test_sub = bbn_test_df[features_to_use]

    bbn = SimpleFairBBN(feature_names=features_to_use, target_name='pass_bar')
    bbn.fit(bbn_train_sub, y_train, include_sensitive=True)

    p_all = bbn.predict_proba(bbn_train_sub).reshape(-1, 1)
    p_race = bbn.predict_proba(bbn_train_sub[['race']]).reshape(-1, 1)
    p_gender = bbn.predict_proba(bbn_train_sub[['gender']]).reshape(-1, 1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_race, p_gender]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_sub).reshape(-1, 1)
    p_race_test = bbn.predict_proba(bbn_test_sub[['race']]).reshape(-1, 1)
    p_gender_test = bbn.predict_proba(bbn_test_sub[['gender']]).reshape(-1, 1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_race_test, p_gender_test]), dtype=torch.float32)

    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    gender_train_t = torch.tensor(sens_gender_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, race_train_t, gender_train_t),
                              batch_size=batch_size, shuffle=True)

    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN_LawSchool(in_dim=32,
                                      race_classes=len(sens_race_train.unique()),
                                      gender_classes=len(sens_gender_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss, total_adv_loss = 0.0, 0.0
        for batch in train_loader:
            batch_in, batch_y, batch_race, batch_gender = batch

            # Train adversary
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, g_logits = adversary(features)
            adv_loss = (adv_loss_fn(r_logits, batch_race) + adv_loss_fn(g_logits, batch_gender)) / 2
            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()

            # Train adapter
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)
            r_logits2, g_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2, batch_race) + adv_loss_fn(g_logits2, batch_gender)) / 2

            grp0_mean = features[batch_race == 0].mean(dim=0)
            grp1_mean = features[batch_race == 1].mean(dim=0)
            dp_penalty = torch.abs(grp0_mean - grp1_mean).mean()

            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward(); adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:03d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")

    adapter.eval()
    with torch.no_grad():
        test_logit, _ = adapter(adapter_in_test, return_features=True)
        test_pred = (torch.sigmoid(test_logit).cpu().numpy().flatten() > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_gender_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_gender_test)
    adv_gender_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_gender_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Gender DP: {base_gender_dp:.4f}, Gender EO: {base_gender_eo:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Gender DP: {adv_gender_dp:.4f}, Gender EO: {adv_gender_eo:.4f}")

    return {
        'baseline': {'pred': base_pred, 'acc': base_acc,
                     'race_dp': base_race_dp, 'race_eo': base_race_eo,
                     'gender_dp': base_gender_dp, 'gender_eo': base_gender_eo},
        'bbn_adapter': {'pred': test_pred, 'acc': adv_acc,
                        'race_dp': adv_race_dp, 'race_eo': adv_race_eo,
                        'gender_dp': adv_gender_dp, 'gender_eo': adv_gender_eo}
    }


if __name__ == '__main__':
    print("Starting Fair BBN System on Kaggle")
    print("=" * 60)
    print("PROCESSING COMPAS DATASET")
    print("=" * 60)
    try:
        compas_data = preprocess_compas_for_fair_bbn()
        compas_results = train_fair_bbn_adapter_compas(compas_data)
        print("✓ COMPAS dataset completed successfully")
    except Exception as e:
        print(f"✗ COMPAS dataset failed: {e}")
        import traceback
        traceback.print_exc()

    print("PROCESSING ADULT INCOME DATASET")
    print("=" * 60)
    try:
        adult_data = preprocess_adult_for_fair_bbn()
        adult_results = train_fair_bbn_adapter_adult(adult_data)
        print("✓ Adult Income dataset completed successfully")
    except Exception as e:
        print(f"✗ Adult Income dataset failed: {e}")
        import traceback
        traceback.print_exc()
    
    print("PROCESSING BANK DATASET")
    print("=" * 60)
    try:
        bank_data = preprocess_bank_for_fair_bbn()
        bank_results = train_fair_bbn_adapter_bank(bank_data)
        print("✓ Bank dataset completed successfully")
    except Exception as e:
        print(f"✗ Bank dataset failed: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "=" * 60)
    print("PROCESSING MEPS DATASET (Synthetic)") 
    print("=" * 60)
    try:
        meps_data = preprocess_meps_for_fair_bbn()
        meps_results = train_fair_bbn_adapter_meps(meps_data)
        print("✓ MEPS dataset completed successfully")
    except Exception as e:
        print(f"✗ MEPS dataset failed: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "=" * 60)
    print("FAIR BBN SYSTEM EXECUTION COMPLETE")
    print("=" * 60)
    print("\n" + "=" * 60)
    print("PROCESSING LAWSCHOOL DATASET")
    print("=" * 60)
    try:
        lawschool_data = preprocess_lawschool_for_fair_bbn()
        lawschool_results = train_fair_bbn_adapter_lawschool(lawschool_data)
        print("✓ LawSchool dataset completed successfully")
    except Exception as e:
        print(f"✗ LawSchool dataset failed: {e}")
        import traceback
        traceback.print_exc()
